In [ ]:
!pip install torch

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


import torch

a = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])

print("Tensor a:")
print(a)

print("\na squared:")
print(a ** 2)


a = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]], requires_grad=True)

print("a:")
print(a)

y = torch.sum(a ** 2)  # 1 + 4 + 9 + 16 = 30
print("\ny:")
print(y)

y.backward()  # compute gradients of y with respect to a

print("\nGradient dy/da:")
print(a.grad)  # should be 2 * a_ij


import torch.nn as nn
import torch.nn.functional as F

class TwoLayerNet(nn.Module):
    def __init__(self, D_in, H, D_out):
        """
        D_in: input dimension
        H: dimension of hidden layer
        D_out: output dimension
        """
        super().__init__()
        self.linear1 = nn.Linear(D_in, H)
        self.linear2 = nn.Linear(H, D_out)

    def forward(self, x):
        """
        x: batch of shape (N, D_in)
        returns: batch of shape (N, D_out)
        """
        h_relu = F.relu(self.linear1(x))
        y_pred = self.linear2(h_relu)
        return y_pred


# N is batch size; D_in is input dimension
# H is hidden dimension; D_out is output dimension
N, D_in, H, D_out = 32, 100, 50, 10

# Random input and labels
x = torch.randn(N, D_in, device=device)             # shape (32, 100)
y = torch.randint(0, D_out, (N,), device=device)    # class labels in [0, 9]

# Construct model and move to device
model = TwoLayerNet(D_in, H, D_out).to(device)

# Forward pass
y_pred = model(x)  # shape (32, 10)

print("Input shape:", x.shape)
print("Output shape:", y_pred.shape)


loss_fn = nn.CrossEntropyLoss()

loss = loss_fn(y_pred, y)

print("Initial loss value:", loss.item())


optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

print("Training for 50 iterations on random data...")

for step in range(50):
    # forward
    y_pred = model(x)
    loss = loss_fn(y_pred, y)

    # backward
    optimizer.zero_grad()
    loss.backward()

    # update
    optimizer.step()

    if (step + 1) % 10 == 0:
        print(f"Step {step + 1:3d} - loss: {loss.item():.4f}")


import torch.nn.functional as F

def my_cross_entropy(outputs, labels):
    batch_size = outputs.size(0)
    log_probs = F.log_softmax(outputs, dim=1)
    selected = log_probs[torch.arange(batch_size), labels]
    return -selected.mean()


ce1 = loss_fn(y_pred, y)
ce2 = my_cross_entropy(y_pred, y)

print("nn.CrossEntropyLoss:", ce1.item())
print("my_cross_entropy:   ", ce2.item())


